In [40]:
import sys

import numpy as np
from transforms3d.axangles import axangle2mat, mat2axangle
import torch
import plotly.graph_objects as go
import trimesh

from mano_pybullet.hand_model import HandModel20

In [41]:
sys.path.append("..")

from utils.grasp_utils import get_handmodel
from model.hand_opt import AdamGraspTransfer

In [42]:
def mat2rvec(mat):
    """Convert rotation matrix to rotation vector."""
    axis, angle = mat2axangle(mat, unit_thresh=1e-05)
    return axis * angle

In [43]:
# NOTE: Set the mano hand models dir here. When using with a script, load this directory from a some config file

%env MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models

env: MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models


## Set Meta Data

In [44]:
# Load data for the 00100 frame
# fname = "sample_hamer_output.npz"

# frame_id = "000172"
frame_id = "000247"

fname = f"{frame_id}.npz"


data = np.load(f"../data/{fname}", allow_pickle=True)

In [45]:
for k in data.keys():
  print(k)

pred_cam
pred_mano_params
pred_cam_t
focal_length
pred_keypoints_3d
pred_vertices
pred_keypoints_2d
opt_translation
bboxes
right
target_transfer_pose


In [46]:
data['right']

array([0, 1])

## Set Left/Right

In [95]:
is_left_hand = True
print("Is left hand? -->", is_left_hand)

rl_index = data['right']
left_idxs = np.arange(rl_index.shape[0])[rl_index==0]
right_idxs = np.arange(rl_index.shape[0])[rl_index==1]

print(left_idxs)
print(right_idxs)

idx_to_use = left_idxs if is_left_hand else right_idxs
print(is_left_hand, idx_to_use)

Is left hand? --> True
[0]
[1]
True [0]


In [96]:
left_idxs.size

1

In [97]:
right_idxs.size

1

In [98]:
mano_params = data['pred_mano_params'].item()

In [99]:
type(mano_params)

dict

In [100]:
mano_params.keys()

dict_keys(['global_orient', 'hand_pose', 'betas'])

In [101]:
mano_params['hand_pose'].shape # for 2 hands

(2, 15, 3, 3)

In [102]:
hand_rotn_mat = mano_params['global_orient'][idx_to_use][0][0]
hand_theta_mat = mano_params['hand_pose'][idx_to_use][0]
mano_trans = data['opt_translation'][idx_to_use][0]
print(hand_rotn_mat.shape)
print(hand_theta_mat.shape)
print(mano_trans.shape)

(3, 3)
(15, 3, 3)
(3,)


In [103]:
hand_theta_full = np.array([mat2rvec(hand_rotn_mat)] + [mat2rvec(hand_theta_mat[i]) for i in range(hand_theta_mat.shape[0])])
print(hand_theta_full.shape)

(16, 3)


In [104]:
hand_model = HandModel20(left_hand=is_left_hand)

In [105]:
angles, palm_basis = hand_model.mano_to_angles(hand_theta_full)

In [106]:
len(angles)

20

In [107]:
palm_basis

array([[-0.1285984 , -0.40649506,  0.90455747],
       [-0.91437279, -0.30450351, -0.26683329],
       [ 0.38390734, -0.86141708, -0.33252936]])

In [108]:
# Reference: https://github.com/kninad/mano_pybullet/blob/960c257cf465f8966e770562b66150beaa359230/mano_pybullet/hand_body.py#L155

origin = hand_model.origins()[0]
palm_trans = mano_trans + origin - palm_basis @ origin


In [109]:
print(palm_trans.shape)

(3,)


## Init Gripper Models

In [110]:
source_gripper = "mano_left" if is_left_hand else "mano_right"
target_gripper = "fetch_gripper"
device = "cpu"

In [111]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [112]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [113]:
# SOURCE GRIPPER (MANO) POSE + DOFS

grasp_pose = torch.zeros(9)
# grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_pose[3:] = torch.tensor(palm_basis.T.reshape(-1)[:6])
grasp_pose[:3] = torch.tensor(palm_trans)
print("Pose:", grasp_pose)

grasp_dofs = torch.tensor(angles) if not is_left_hand else -1 * torch.tensor(angles)

# grasp_dofs = torch.tensor(angles)

print("DOFS:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

Pose: tensor([ 0.7320,  0.2914,  1.3866, -0.1286, -0.9144,  0.3839, -0.4065, -0.3045,
        -0.8614])
DOFS: tensor([-0.5086,  0.4347,  0.7161,  0.5566, -0.1119,  0.6991,  0.7083,  0.3568,
         0.6774,  0.2246,  0.3012,  0.1018,  0.1619,  0.5852,  0.5672,  0.3419,
        -1.1638, -0.5251,  0.4238, -0.3669])


In [114]:
source_model.dynamic_joints_q_lower.squeeze(0)

tensor([-0.3491, -0.1745,  0.0000,  0.0000, -0.5236, -0.1745,  0.0000,  0.0000,
        -0.6981, -0.1745,  0.0000,  0.0000, -0.5236, -0.1745,  0.0000,  0.0000,
        -0.1745, -0.6981,  0.0000,  0.0000])

In [115]:
source_model.dynamic_joints_q_upper.squeeze(0)

tensor([0.3491, 1.5708, 1.7453, 1.7453, 0.3491, 1.5708, 1.7453, 1.7453, 0.3491,
        1.5708, 1.7453, 1.7453, 0.3491, 1.5708, 1.7453, 1.7453, 2.6180, 0.6981,
        1.7453, 1.7453])

In [116]:
grasp_dofs

tensor([-0.5086,  0.4347,  0.7161,  0.5566, -0.1119,  0.6991,  0.7083,  0.3568,
         0.6774,  0.2246,  0.3012,  0.1018,  0.1619,  0.5852,  0.5672,  0.3419,
        -1.1638, -0.5251,  0.4238, -0.3669])

In [117]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

In [118]:
q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

In [119]:
print(q_traj.shape)
best_q = q_traj[0, -1]
print(best_q.shape)

torch.Size([32, 301, 9])
torch.Size([9])


In [120]:
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat((best_q, target_model.dynamic_joints_q_upper[0]), dim=0)

## Viz Src + Target

In [121]:
print("Plotting TARGET and SOURCE together...")

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red')
target_gripper_mesh_data = target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='green')
vis_data += target_gripper_mesh_data
fig = go.Figure(data=vis_data)
fig.show()
# fig.write_html("../logs_viz/gtransfer_test.html")


Plotting TARGET and SOURCE together...


In [122]:
len(target_gripper_mesh_data)

3

In [123]:
trimesh_list = []

for mesh in target_gripper_mesh_data:
    vertices = np.array([mesh.x, mesh.y, mesh.z]).T
    faces = np.array([mesh.i, mesh.j, mesh.k]).T
    trimesh_list.append(trimesh.Trimesh(vertices=vertices, faces=faces))

combined_mesh = trimesh.util.concatenate(trimesh_list)
combined_mesh.export('mesh.ply')

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 987\nproperty float x\nproperty float y\nproperty float z\nelement face 1962\nproperty list uchar int vertex_indices\nend_header\n\x138<?\xfb\x9f\xb1>A7\xa1?\xdd\x8a<?\xd85\xb1>\xd7/\xa1?\x818:?\x98d\xa9>&\xa5\xa1?\x04\x15B?\xc9\xaf\xa9>\xc5\xdb\xa0?\xd0\x17B?\xb6\xc8\xa9>\xca\xda\xa0?\xf3\xdc@?\\\x11\x9d>\xe1]\xa1?\x01\x85.?\x1a\xe2\x81>p\x01\xa4?\xd7r.?\xe1\x9d\x81>\xf1\n\xa4?\xe2X.?\xa0\xdd\x82>\x7f\xf4\xa3?\xc6\xb83?\nir>\xf6\x8e\xa5?\x82\xe53?4\x92\x83>\xfcz\xb1?\x0bd.?6\x94~>$\xf1\xa5?-<9?k\x95z>\xae\xe0\xb0?\x8f\x064?\x03\xd1q>\xa0\x87\xa5?D\x9e<?\xcfO\x8a>\xd4\xc4\xa2?W\xbf9?\xd5{\x83>^\x00\xa3?\xeee<?\xb0`\x8a>\xdf\x88\xa2?\x1c\x17,?\xe8\xaa\xa0>\xf8\n\xa4?.1-?w\xa0\xa4>\xea\x06\xa4?\'&-?E~\xa4>9\xd4\xa3?xL.?\xf13\xa8>\xe5\x9a\xa3?\xb8\xc9+?\x0b\xcd\x9b>\xcd\xd6\xa3?\xf9\x90(?$Q\x8d>\xd7\x8a\xa4?k>(?\xc9\x98\x8d>*\x9e\xa4?\t;\'?\xe0H\x8d>\xf4e\xa6?\xee=\'?\x8d\\\x8d>W\x9b\xa6?\x1b

## Viz Mano + URDF

In [124]:
hand_ply = f"{frame_id}_{int(not is_left_hand)}.ply"

mano_mesh = trimesh.load_mesh(f"../data/{hand_ply}")


x, y, z = mano_mesh.vertices.T
# i, j, k = mano_mesh.faces.T

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)
vis_data += [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2, color='green')
        )
    ]


fig = go.Figure(data=vis_data)
fig.show()
# fig.write_html("../logs_viz/gtransfer_test.html")
